# NLP multilabel classification: robust ensemble v2 (head-tail + ASL + stable thresholds)


In [ ]:
import os
import re
import ast
import html
import unicodedata
import random
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 322

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

try:
    import torch

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

print("SEED:", SEED)


## 1. Загрузка данных

Ноутбук рассчитан на запуск из корня репозитория.  
Ожидаемые файлы:
- `train.csv`
- `test.csv` или `test/test.csv`
- `sample_submission.csv`

In [ ]:
# Поиск файлов в типовых местах

def find_file(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError(f"Файл не найден. Проверенные пути: {candidates}")

TRAIN_PATH = find_file([
    "train.csv",
    "train/train.csv",
    "data/train.csv",
    "input/train.csv",
])

TEST_PATH = find_file([
    "test.csv",
    "test/test.csv",
    "data/test.csv",
    "input/test.csv",
])

SAMPLE_PATH = find_file([
    "sample_submission.csv",
    "data/sample_submission.csv",
    "input/sample_submission.csv",
])

train = pd.read_csv(TRAIN_PATH, sep="\t")
test = pd.read_csv(TEST_PATH, sep="\t")
sample_submission = pd.read_csv(SAMPLE_PATH, sep=",")

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("SAMPLE_PATH:", SAMPLE_PATH)
print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

## 2. Проверка структуры данных

In [ ]:
required_train_cols = {"id", "source", "title", "text", "publication_date", "target"}
required_test_cols = {"id", "source", "title", "text", "publication_date"}
required_submission_cols = {"id", "target"}

assert required_train_cols.issubset(train.columns), f"В train не хватает колонок: {required_train_cols - set(train.columns)}"
assert required_test_cols.issubset(test.columns), f"В test не хватает колонок: {required_test_cols - set(test.columns)}"
assert required_submission_cols.issubset(sample_submission.columns), f"В sample_submission не хватает колонок: {required_submission_cols - set(sample_submission.columns)}"

assert len(test) == len(sample_submission), "Размер test и sample_submission не совпадает"
assert set(sample_submission["id"]) == set(test["id"]), "id в test и sample_submission не совпадают"

print("Структура файлов корректна.")

## 3. Разбор target

`target` должен превратиться из строки вида `"[0,0,1,0,1]"` в массив из 5 чисел.

In [ ]:
def parse_target(value):
    """
    Преобразует target в список из 5 int.
    Поддерживает строки вида:
    - "[0,0,1,0,1]"
    - "[0, 0, 1, 0, 1]"
    - уже готовые list/tuple/np.ndarray
    """
    if isinstance(value, (list, tuple, np.ndarray)):
        parsed = list(value)
    else:
        value = str(value).strip()
        try:
            parsed = ast.literal_eval(value)
        except Exception:
            # fallback на случай нестандартной строки
            nums = re.findall(r"-?\d+", value)
            parsed = [int(x) for x in nums]

    parsed = [int(x) for x in parsed]
    if len(parsed) != 5:
        raise ValueError(f"Ожидалось 5 меток, получено {len(parsed)}: {value}")
    return parsed

target_lists = train["target"].apply(parse_target)
y = np.array(target_lists.tolist(), dtype=int)

print("y shape:", y.shape)
print("Первые target:")
display(pd.DataFrame(y, columns=[f"label_{i}" for i in range(y.shape[1])]).head())

assert y.shape[1] == 5
assert set(np.unique(y)).issubset({0, 1})

## 4. EDA

Минимальный анализ, который нужен для понимания данных и для полноценности решения:
- пропуски;
- баланс классов;
- количество меток на объект;
- длины текстов;
- распределение источников.

In [ ]:
print("Пропуски в train:")
display(train.isna().sum())

print("Пропуски в test:")
display(test.isna().sum())

label_cols = [f"label_{i}" for i in range(5)]
target_df = pd.DataFrame(y, columns=label_cols)

print("Доля положительных примеров по каждому классу:")
display(target_df.mean().to_frame("positive_rate"))

print("Количество положительных примеров по каждому классу:")
display(target_df.sum().to_frame("positive_count"))

print("Распределение числа меток на одну новость:")
display(target_df.sum(axis=1).value_counts().sort_index().to_frame("count"))

print("Топ источников train:")
display(train["source"].fillna("missing").value_counts().head(20).to_frame("count"))

tmp_lengths = pd.DataFrame({
    "title_len": train["title"].fillna("").astype(str).str.len(),
    "text_len": train["text"].fillna("").astype(str).str.len(),
})
print("Длины title/text:")
display(tmp_lengths.describe())

## 5. Предобработка текста

Используем только поля из датасета.  
`source` и дата добавляются прямо в текст как дополнительные токены — это простой способ учесть категориальный источник и время публикации без отдельного сложного feature engineering.

In [ ]:
def clean_text(value):
    """
    Жёсткая очистка для TF-IDF.

    Учитываем реальные паттерны датасета:
    - HTML/XML/WP-теги;
    - HTML-сущности;
    - URL/email;
    - emoji;
    - проценты и десятичные числа;
    - валюты;
    - hashtags и mentions;
    - JSON-вставки с картинками;
    - длинные телефонные/донатные номера.
    """
    if pd.isna(value):
        return ""

    value = str(value)
    value = unicodedata.normalize("NFKC", value)
    value = value.replace("\xa0", " ")

    # Удаляем комментарии WordPress/HTML и XML/CDATA-маркеры ДО html.unescape
    value = re.sub(r"<!--.*?-->", " ", value, flags=re.DOTALL)
    value = re.sub(r"<\?xml[^>]*\?>", " ", value, flags=re.IGNORECASE)
    value = re.sub(r"<!\[CDATA\[|]]>", " ", value)

    # Удаляем настоящие HTML-теги ДО html.unescape
    value = re.sub(
        r"</?(?:p|br|span|div|content|title|text|item|section|body|html|head|i|b|strong|em|u|li|ul|ol|blockquote|hr|a|img|figure|figcaption|h[1-6])\b[^>]*>",
        " ",
        value,
        flags=re.IGNORECASE
    )

    # Теперь безопасно декодируем HTML-сущности
    value = html.unescape(value)
    value = value.replace("\xa0", " ")

    # JSON-вставки с картинками
    value = re.sub(r'\[\s*\{.*?"image".*?\}\s*\]', " image_token ", value, flags=re.DOTALL)
    value = re.sub(r'"uuid"\s*:\s*"[^"]+"', " image_token ", value)

    # URL и email
    value = re.sub(r"https?://\S+|www\.\S+|t\.me/\S+", " url_token ", value, flags=re.IGNORECASE)
    value = re.sub(r"\S+@\S+", " email_token ", value)

    # Hashtags: #covid19 -> hashtag_covid19
    value = re.sub(r"#([\wА-Яа-яЁё_]+)", r" hashtag_\1 ", value)

    # Mentions: @navalny -> mention_token navalny
    value = re.sub(r"(?<!\w)@([\w_]+)", r" mention_token \1 ", value)

    # Emoji / служебные unicode-символы
    value = re.sub(
        r"[\U0001F300-\U0001FAFF\U00002700-\U000027BF\U00002600-\U000026FF]",
        " ",
        value
    )
    value = re.sub(r"[\u200b-\u200f\u202a-\u202e\ufeff]", " ", value)

    # Базовая нормализация
    value = value.lower()
    value = value.replace("ё", "е")

    # Валюты
    value = value.replace("₽", " рубль ")
    value = value.replace("$", " доллар ")
    value = value.replace("€", " евро ")
    value = value.replace("£", " фунт ")
    value = value.replace("¥", " юань ")

    # Десятичные числа: 19,4 или 19.4 -> 19_4
    value = re.sub(r"(?<=\d)[,.](?=\d)", "_", value)

    # Проценты: 19_4% -> 19_4 процент
    value = re.sub(r"%", " процент ", value)

    # Длинные номера телефонов/кошельков/донатов: не даём им распасться на шумные числа
    value = re.sub(r"\+?\d[\d\s\-\(\)]{8,}\d", " long_number_token ", value)

    # Диапазоны: 5-8 -> 5_8_range
    value = re.sub(r"\b(\d+)\s*[-–—]\s*(\d+)\b", r" \1_\2_range ", value)

    # Остальную пунктуацию убираем
    value = re.sub(r"[^0-9a-zа-я_\s]", " ", value)

    # Нормализация пробелов
    value = re.sub(r"[\r\n\t]+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def clean_text_for_transformer(value):
    """
    Мягкая очистка для transformer.
    Пунктуацию полностью не убираем: tokenizer сам её обработает.
    """
    if pd.isna(value):
        return ""

    value = str(value)
    value = unicodedata.normalize("NFKC", value)
    value = value.replace("\xa0", " ")

    # Удаляем HTML/XML/WP-мусор ДО html.unescape
    value = re.sub(r"<!--.*?-->", " ", value, flags=re.DOTALL)
    value = re.sub(r"<\?xml[^>]*\?>", " ", value, flags=re.IGNORECASE)
    value = re.sub(r"<!\[CDATA\[|]]>", " ", value)

    value = re.sub(
        r"</?(?:p|br|span|div|content|title|text|item|section|body|html|head|i|b|strong|em|u|li|ul|ol|blockquote|hr|a|img|figure|figcaption|h[1-6])\b[^>]*>",
        " ",
        value,
        flags=re.IGNORECASE
    )

    value = html.unescape(value)
    value = value.replace("\xa0", " ")

    # JSON-вставки с картинками
    value = re.sub(r'\[\s*\{.*?"image".*?\}\s*\]', " IMAGE_TOKEN ", value, flags=re.DOTALL)
    value = re.sub(r'"uuid"\s*:\s*"[^"]+"', " IMAGE_TOKEN ", value)

    # URL/email
    value = re.sub(r"https?://\S+|www\.\S+|t\.me/\S+", " URL_TOKEN ", value, flags=re.IGNORECASE)
    value = re.sub(r"\S+@\S+", " EMAIL_TOKEN ", value)

    # Emoji / служебные символы
    value = re.sub(
        r"[\U0001F300-\U0001FAFF\U00002700-\U000027BF\U00002600-\U000026FF]",
        " ",
        value
    )
    value = re.sub(r"[\u200b-\u200f\u202a-\u202e\ufeff]", " ", value)

    value = re.sub(r"[\r\n\t]+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def clean_source_token(value):
    value = clean_text(value)
    value = re.sub(r"\s+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")

    if value == "":
        value = "missing_source"

    return "source_" + value


def prepare_datetime_features(df):
    dates = pd.to_datetime(df["publication_date"], errors="coerce")

    out = pd.DataFrame(index=df.index)
    out["year"] = dates.dt.year.fillna(-1).astype(int).astype(str)
    out["month"] = dates.dt.month.fillna(-1).astype(int).astype(str)
    out["dayofweek"] = dates.dt.dayofweek.fillna(-1).astype(int).astype(str)

    return out


def build_joined_tfidf_text(df):
    """
    Один общий текст для TF-IDF.
    Заголовок дублируем, потому что он часто сильнее основного текста.
    """
    date_features = prepare_datetime_features(df)

    source = df["source"].fillna("missing_source").astype(str).map(clean_source_token)
    title = df["title"].fillna("").astype(str).map(clean_text)
    text = df["text"].fillna("").astype(str).map(clean_text)

    full_text = (
        source + " " +
        "year_" + date_features["year"] + " " +
        "month_" + date_features["month"] + " " +
        "dow_" + date_features["dayofweek"] + " " +
        "title " + title + " " +
        "title " + title + " " +
        "title " + title + " " +
        "text " + text
    )

    return full_text


def build_split_tfidf_df(df):
    """
    Раздельные признаки для title / text / meta.
    """
    date_features = prepare_datetime_features(df)

    source = df["source"].fillna("missing_source").astype(str).map(clean_source_token)
    title = df["title"].fillna("").astype(str).map(clean_text)
    text = df["text"].fillna("").astype(str).map(clean_text)

    meta = (
        source + " " +
        "year_" + date_features["year"] + " " +
        "month_" + date_features["month"] + " " +
        "dow_" + date_features["dayofweek"]
    )

    return pd.DataFrame({
        "title_clean": title,
        "text_clean": text,
        "meta_clean": meta,
    })


def build_transformer_text(df, head_chars=2600, tail_chars=2200):
    """
    Текст для transformer.
    Используем head+tail, а не только начало текста: у длинных материалов важные маркеры класса
    могут находиться в конце. Source/date намеренно не добавляем во вход transformer.
    """
    title = df["title"].fillna("").astype(str).map(clean_text_for_transformer)
    text = df["text"].fillna("").astype(str).map(clean_text_for_transformer)

    head = text.str[:head_chars]
    tail = text.str[-tail_chars:]
    tail = tail.where(text.str.len() > head_chars + tail_chars, "")

    full_text = (
        "Заголовок: " + title + ". " +
        "Заголовок: " + title + ". " +
        "Начало текста: " + head + ". " +
        "Конец текста: " + tail
    )

    return full_text


def build_joined_tfidf_text_nometa(df):
    """
    TF-IDF текст без source/date.
    Нужен как анти-overfit вариант для ансамбля.
    """
    title = df["title"].fillna("").astype(str).map(clean_text)
    text = df["text"].fillna("").astype(str).map(clean_text)

    full_text = (
        "title " + title + " " +
        "title " + title + " " +
        "text " + text
    )

    return full_text


X_text = build_joined_tfidf_text(train)
X_test_text = build_joined_tfidf_text(test)

X_split = build_split_tfidf_df(train)
X_test_split = build_split_tfidf_df(test)

X_transformer_text = build_transformer_text(train)
X_test_transformer_text = build_transformer_text(test)

X_text_nometa = build_joined_tfidf_text_nometa(train)
X_test_text_nometa = build_joined_tfidf_text_nometa(test)

print("TF-IDF example:")
print(X_text.iloc[0][:700])

print("\nTransformer example:")
print(X_transformer_text.iloc[0][:700])


## 6. Cross-validation схема

Для финального решения используется 5-fold cross-validation.  
OOF-предсказания нужны не только для честной локальной оценки, но и для обучения второй нейросети-корректора без утечки target.



In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import hamming_loss, f1_score, accuracy_score

Y = y.astype(np.float32)
N_LABELS = Y.shape[1]
N_SPLITS = 5


def make_stratification_key(y_array, n_splits=5):
    patterns = pd.Series(["".join(map(str, row.astype(int))) for row in y_array])
    counts = patterns.value_counts()
    strat_key = patterns.where(patterns.map(counts) >= n_splits, "rare")

    if strat_key.value_counts().min() < n_splits:
        return None

    return strat_key.values


def make_cv_splits(y_array, n_splits=5, seed=SEED):
    strat_key = make_stratification_key(y_array, n_splits=n_splits)
    indices = np.arange(len(y_array))

    if strat_key is not None:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        print("CV: StratifiedKFold по комбинациям target.")
        return list(splitter.split(indices, strat_key))

    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    print("CV: KFold, потому что часть комбинаций target редкая.")
    return list(splitter.split(indices))


def hamming_score(y_true, y_pred):
    return 1.0 - hamming_loss(y_true, y_pred)


def print_metrics(y_true, y_pred, title="metrics"):
    print(title)
    print("hamming_loss :", hamming_loss(y_true, y_pred))
    print("hamming_score:", hamming_score(y_true, y_pred))
    print("micro_f1     :", f1_score(y_true, y_pred, average="micro", zero_division=0))
    print("macro_f1     :", f1_score(y_true, y_pred, average="macro", zero_division=0))
    print("samples_f1   :", f1_score(y_true, y_pred, average="samples", zero_division=0))
    print("subset_acc   :", accuracy_score(y_true, y_pred))


cv_splits = make_cv_splits(Y, n_splits=N_SPLITS, seed=SEED)

print("Количество folds:", len(cv_splits))
for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print(f"fold {fold}: train={len(train_idx)}, valid={len(valid_idx)}")



## 7. Модельная схема

Итоговая схема сделана как честный OOF-ансамбль:

1. `ai-forever/ruRoberta-large` — основная сильная transformer-модель.
2. `DeepPavlov/rubert-base-cased` — вторая независимая нейросеть для снижения ошибки ансамбля.
3. TF-IDF + Logistic Regression / ComplementNB — классические методы ГО, которые часто ловят устойчивые n-граммные паттерны.
4. MLP-корректор — обучается поверх OOF-предсказаний базовых моделей, SVD-признаков и метаданных.
5. Финальный label-wise ensemble — для каждого класса отдельно подбирает веса моделей и точный порог по OOF.


In [ ]:
import importlib
import subprocess
import gc

if importlib.util.find_spec("transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    DATALOADER_NUM_WORKERS = 2
else:
    gpu_name = "CPU"
    gpu_mem_gb = 0
    DATALOADER_NUM_WORKERS = 0

print("DEVICE:", DEVICE)
print("GPU:", gpu_name)
print("GPU memory, GB:", round(gpu_mem_gb, 2))


def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass


def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def prob_to_logit_np(p, eps=1e-5):
    p = np.clip(np.asarray(p, dtype=np.float32), eps, 1.0 - eps)
    return np.log(p / (1.0 - p)).astype(np.float32)


def make_pos_weight(labels, mode=None):
    if mode is None:
        return None

    labels = np.asarray(labels, dtype=np.float32)
    pos = labels.sum(axis=0)
    neg = labels.shape[0] - pos
    weight = neg / np.maximum(pos, 1.0)

    if mode == "sqrt":
        weight = np.sqrt(weight)
    elif mode == "log":
        weight = np.log1p(weight)
    elif mode != "full":
        raise ValueError(f"Неизвестный режим pos_weight: {mode}")

    weight = np.clip(weight, 1.0, 4.0).astype(np.float32)
    return torch.tensor(weight, dtype=torch.float32, device=DEVICE)


class AsymmetricLoss(torch.nn.Module):
    """
    Asymmetric Loss для multilabel-классификации.
    Она мягче штрафует уверенные отрицательные классы и сильнее фокусируется на сложных примерах.
    Это обычно устойчивее BCE при дисбалансе меток.
    """
    def __init__(self, gamma_neg=3.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        targets = targets.float()
        prob = torch.sigmoid(logits)

        prob_pos = prob
        prob_neg = 1.0 - prob

        if self.clip is not None and self.clip > 0:
            prob_neg = torch.clamp(prob_neg + self.clip, max=1.0)

        loss_pos = targets * torch.log(torch.clamp(prob_pos, min=self.eps))
        loss_neg = (1.0 - targets) * torch.log(torch.clamp(prob_neg, min=self.eps))

        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt = prob_pos * targets + prob_neg * (1.0 - targets)
            gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            focal_weight = torch.pow(1.0 - pt, gamma)
            loss_pos = loss_pos * focal_weight
            loss_neg = loss_neg * focal_weight

        return -(loss_pos + loss_neg).mean()


class TokenizedTransformerDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length, labels=None):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        self.labels = None if labels is None else np.asarray(labels, dtype=np.float32)

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item


def make_transformer_collate_fn(tokenizer):
    def collate_fn(batch):
        labels = None
        if "labels" in batch[0]:
            labels = [example.pop("labels") for example in batch]

        padded = tokenizer.pad(batch, padding=True, return_tensors="pt")

        if labels is not None:
            padded["labels"] = torch.tensor(np.asarray(labels), dtype=torch.float32)

        return padded

    return collate_fn


def make_train_loader(dataset, tokenizer, batch_size, num_workers):
    generator = torch.Generator()
    generator.manual_seed(SEED)

    loader_kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=make_transformer_collate_fn(tokenizer),
        drop_last=False,
    )

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True

    return DataLoader(**loader_kwargs)


def make_predict_loader(dataset, tokenizer, batch_size, num_workers):
    loader_kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=make_transformer_collate_fn(tokenizer),
        drop_last=False,
    )

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True

    return DataLoader(**loader_kwargs)


def make_adamw(parameters, lr, weight_decay):
    if torch.cuda.is_available():
        try:
            return torch.optim.AdamW(parameters, lr=lr, weight_decay=weight_decay, fused=True)
        except TypeError:
            pass

    return torch.optim.AdamW(parameters, lr=lr, weight_decay=weight_decay)


def auto_batch_params(model_size, max_length, effective_batch_size):
    if not torch.cuda.is_available():
        train_batch = 2
        predict_batch = 8
    elif gpu_mem_gb >= 35:
        if model_size == "large":
            train_batch = 12 if max_length <= 320 else 8
            predict_batch = 48 if max_length <= 320 else 32
        else:
            train_batch = 24 if max_length <= 320 else 16
            predict_batch = 96 if max_length <= 320 else 64
    elif gpu_mem_gb >= 20:
        if model_size == "large":
            train_batch = 6 if max_length <= 320 else 4
            predict_batch = 32 if max_length <= 320 else 24
        else:
            train_batch = 12 if max_length <= 320 else 8
            predict_batch = 64 if max_length <= 320 else 48
    else:
        if model_size == "large":
            train_batch = 3 if max_length <= 320 else 2
            predict_batch = 16
        else:
            train_batch = 6 if max_length <= 320 else 4
            predict_batch = 32

    grad_accum_steps = max(1, int(np.ceil(effective_batch_size / train_batch)))
    return train_batch, predict_batch, grad_accum_steps


class TransformerMultilabelClassifier:
    def __init__(
        self,
        model_name,
        num_labels=5,
        max_length=320,
        epochs=3,
        batch_size=8,
        grad_accum_steps=2,
        predict_batch_size=32,
        lr=1.5e-5,
        weight_decay=0.01,
        warmup_ratio=0.08,
        num_workers=0,
        fp16=True,
        pos_weight_mode=None,
        loss_name="bce",
    ):
        self.model_name = model_name
        self.num_labels = num_labels
        self.max_length = max_length
        self.epochs = epochs
        self.batch_size = batch_size
        self.grad_accum_steps = grad_accum_steps
        self.predict_batch_size = predict_batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.warmup_ratio = warmup_ratio
        self.num_workers = num_workers
        self.fp16 = fp16 and torch.cuda.is_available()
        self.pos_weight_mode = pos_weight_mode
        self.loss_name = loss_name

        self.tokenizer = None
        self.model = None
        self.loss_fn = None
        self.best_epoch_ = epochs

    def _make_loss(self, labels):
        if self.loss_name == "asl":
            return AsymmetricLoss(gamma_neg=3.0, gamma_pos=0.0, clip=0.05)

        pos_weight = make_pos_weight(labels, mode=self.pos_weight_mode)
        return torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def fit(self, texts, labels, valid_texts=None, valid_labels=None):
        seed_everything(SEED)

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="multi_label_classification",
            ignore_mismatched_sizes=True,
        ).to(DEVICE)

        self.loss_fn = self._make_loss(labels)

        train_dataset = TokenizedTransformerDataset(
            texts=texts,
            tokenizer=self.tokenizer,
            max_length=self.max_length,
            labels=labels,
        )

        train_loader = make_train_loader(
            dataset=train_dataset,
            tokenizer=self.tokenizer,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
        )

        valid_loader = None
        if valid_texts is not None and valid_labels is not None:
            valid_dataset = TokenizedTransformerDataset(
                texts=valid_texts,
                tokenizer=self.tokenizer,
                max_length=self.max_length,
                labels=valid_labels,
            )
            valid_loader = make_predict_loader(
                dataset=valid_dataset,
                tokenizer=self.tokenizer,
                batch_size=self.predict_batch_size,
                num_workers=self.num_workers,
            )

        optimizer = make_adamw(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        update_steps_per_epoch = int(np.ceil(len(train_loader) / self.grad_accum_steps))
        total_steps = update_steps_per_epoch * self.epochs
        warmup_steps = int(total_steps * self.warmup_ratio)

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        scaler = torch.cuda.amp.GradScaler(enabled=self.fp16)
        best_state = None
        best_valid_loss = np.inf

        optimizer.zero_grad(set_to_none=True)

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            losses = []

            for step, batch in enumerate(train_loader, start=1):
                labels_batch = batch.pop("labels").to(DEVICE, non_blocking=True)
                batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

                with torch.cuda.amp.autocast(enabled=self.fp16):
                    logits = self.model(**batch).logits
                    loss = self.loss_fn(logits, labels_batch)
                    loss = loss / self.grad_accum_steps

                scaler.scale(loss).backward()
                losses.append(loss.item() * self.grad_accum_steps)

                is_update_step = (step % self.grad_accum_steps == 0) or (step == len(train_loader))
                if is_update_step:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            if valid_loader is not None:
                valid_loss = self.evaluate_loss(valid_loader)
                print(f"epoch {epoch}/{self.epochs} | train_loss: {np.mean(losses):.6f} | valid_loss: {valid_loss:.6f}")

                if valid_loss < best_valid_loss:
                    best_valid_loss = valid_loss
                    self.best_epoch_ = epoch
                    best_state = {
                        key: value.detach().cpu().clone()
                        for key, value in self.model.state_dict().items()
                    }
            else:
                print(f"epoch {epoch}/{self.epochs} | train_loss: {np.mean(losses):.6f}")

        if best_state is not None:
            self.model.load_state_dict(best_state)
            del best_state

        self.model.eval()
        return self

    def evaluate_loss(self, loader):
        self.model.eval()
        losses = []

        with torch.no_grad():
            for batch in loader:
                labels_batch = batch.pop("labels").to(DEVICE, non_blocking=True)
                batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

                with torch.cuda.amp.autocast(enabled=self.fp16):
                    logits = self.model(**batch).logits
                    loss = self.loss_fn(logits, labels_batch)

                losses.append(loss.item())

        return float(np.mean(losses))

    def predict_logits(self, texts):
        dataset = TokenizedTransformerDataset(
            texts=texts,
            tokenizer=self.tokenizer,
            max_length=self.max_length,
            labels=None,
        )

        loader = make_predict_loader(
            dataset=dataset,
            tokenizer=self.tokenizer,
            batch_size=self.predict_batch_size,
            num_workers=self.num_workers,
        )

        all_logits = []
        self.model.eval()

        with torch.no_grad():
            for batch in loader:
                batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

                with torch.cuda.amp.autocast(enabled=self.fp16):
                    logits = self.model(**batch).logits

                all_logits.append(logits.detach().cpu().numpy().astype(np.float32))

        return np.vstack(all_logits)


## 8. OOF-обучение двух transformer-моделей

Каждая нейросеть обучается на одинаковых fold-разбиениях. Для train-части сохраняются только OOF-предсказания, поэтому последующий подбор весов и порогов не видит target той строки, для которой строится прогноз.

Для финального test-прогноза каждая transformer-модель дообучается на всём train с числом эпох, выбранным по median best epoch на OOF.


In [ ]:
seed_everything(SEED)

# Две независимые нейросети: сильная RuRoBERTa-large и более лёгкая RuBERT-base.
# Они обучаются на одной и той же CV-схеме, поэтому их OOF-предсказания можно честно ансамблировать.
TRANSFORMER_CONFIGS = [
    {
        "name": "ruroberta_large_asl_512",
        "model_name": "ai-forever/ruRoberta-large",
        "model_size": "large",
        "max_length": 512,
        "epochs": 3,
        "effective_batch_size": 16,
        "lr": 1.1e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.08,
        "pos_weight_mode": None,
        "loss_name": "asl",
    },
    {
        "name": "xlm_roberta_large_asl_512",
        "model_name": "FacebookAI/xlm-roberta-large",
        "model_size": "large",
        "max_length": 512,
        "epochs": 3,
        "effective_batch_size": 16,
        "lr": 1.0e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.08,
        "pos_weight_mode": None,
        "loss_name": "asl",
    },
]


def run_transformer_cv(config):
    train_batch, predict_batch, grad_accum_steps = auto_batch_params(
        model_size=config["model_size"],
        max_length=config["max_length"],
        effective_batch_size=config["effective_batch_size"],
    )

    print("=" * 100)
    print("Transformer:", config["name"])
    print("model:", config["model_name"])
    print("max_length:", config["max_length"])
    print("epochs:", config["epochs"])
    print("train_batch:", train_batch)
    print("grad_accum_steps:", grad_accum_steps)
    print("effective_batch:", train_batch * grad_accum_steps)
    print("predict_batch:", predict_batch)
    print("pos_weight_mode:", config["pos_weight_mode"])
    print("loss_name:", config.get("loss_name", "bce"))

    oof_logits = np.zeros((len(train), N_LABELS), dtype=np.float32)
    oof_probs = np.zeros((len(train), N_LABELS), dtype=np.float32)
    best_epochs = []

    for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        print("=" * 80)
        print(f"{config['name']} fold {fold}/{N_SPLITS}")

        clf = TransformerMultilabelClassifier(
            model_name=config["model_name"],
            num_labels=N_LABELS,
            max_length=config["max_length"],
            epochs=config["epochs"],
            batch_size=train_batch,
            grad_accum_steps=grad_accum_steps,
            predict_batch_size=predict_batch,
            lr=config["lr"],
            weight_decay=config["weight_decay"],
            warmup_ratio=config["warmup_ratio"],
            num_workers=DATALOADER_NUM_WORKERS,
            fp16=True,
            pos_weight_mode=config["pos_weight_mode"],
            loss_name=config.get("loss_name", "bce"),
        )

        clf.fit(
            X_transformer_text.iloc[train_idx].values,
            Y[train_idx],
            valid_texts=X_transformer_text.iloc[valid_idx].values,
            valid_labels=Y[valid_idx],
        )

        valid_logits = clf.predict_logits(X_transformer_text.iloc[valid_idx].values)
        valid_probs = sigmoid_np(valid_logits)

        oof_logits[valid_idx] = valid_logits
        oof_probs[valid_idx] = valid_probs
        best_epochs.append(clf.best_epoch_)

        fold_pred = (valid_probs >= 0.5).astype(int)
        print("best_epoch:", clf.best_epoch_)
        print_metrics(Y[valid_idx], fold_pred, title=f"{config['name']} fold {fold} @0.5")

        del clf, valid_logits, valid_probs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    assert not np.isnan(oof_probs).any()
    print_metrics(Y, (oof_probs >= 0.5).astype(int), title=f"{config['name']} OOF @0.5")

    final_epochs = int(np.median(best_epochs))
    final_epochs = max(2, min(config["epochs"], final_epochs))
    print(f"{config['name']} final epochs:", final_epochs)

    final_clf = TransformerMultilabelClassifier(
        model_name=config["model_name"],
        num_labels=N_LABELS,
        max_length=config["max_length"],
        epochs=final_epochs,
        batch_size=train_batch,
        grad_accum_steps=grad_accum_steps,
        predict_batch_size=predict_batch,
        lr=config["lr"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],
        num_workers=DATALOADER_NUM_WORKERS,
        fp16=True,
        pos_weight_mode=config["pos_weight_mode"],
        loss_name=config.get("loss_name", "bce"),
    )

    final_clf.fit(X_transformer_text.values, Y)
    test_logits = final_clf.predict_logits(X_test_transformer_text.values)
    test_probs = sigmoid_np(test_logits)

    del final_clf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "oof_logits": oof_logits,
        "oof_probs": oof_probs,
        "test_logits": test_logits.astype(np.float32),
        "test_probs": test_probs.astype(np.float32),
        "best_epochs": best_epochs,
        "final_epochs": final_epochs,
    }


transformer_outputs = {}
for cfg in TRANSFORMER_CONFIGS:
    transformer_outputs[cfg["name"]] = run_transformer_cv(cfg)

# Переменные ниже оставлены для совместимости с последующими блоками.
primary_transformer_name = TRANSFORMER_CONFIGS[0]["name"]
transformer_oof_logits = transformer_outputs[primary_transformer_name]["oof_logits"]
transformer_oof_probs = transformer_outputs[primary_transformer_name]["oof_probs"]
transformer_test_logits = transformer_outputs[primary_transformer_name]["test_logits"]
transformer_test_probs = transformer_outputs[primary_transformer_name]["test_probs"]

model_oof_probs = {
    f"tr_{name}": value["oof_probs"]
    for name, value in transformer_outputs.items()
}
model_test_probs = {
    f"tr_{name}": value["test_probs"]
    for name, value in transformer_outputs.items()
}

print("Base neural components:", list(model_oof_probs.keys()))


## 9. TF-IDF-модели и признаки для MLP-корректора

Добавляем независимый классический блок:

- TF-IDF word/char n-граммы с метаданными;
- TF-IDF без source/date как менее overfit-вариант;
- ComplementNB как отдельная вероятностная модель для текстовых n-грамм.

После этого собираем матрицу для MLP-корректора: вероятности и logits всех базовых моделей + SVD по TF-IDF + простые метапризнаки.


In [ ]:
from sklearn.base import clone
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from scipy import sparse


def make_tfidf_svd_features(train_texts, test_texts, n_components=320):
    tfidf = FeatureUnion([
        (
            "word",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=140_000,
                sublinear_tf=True,
                dtype=np.float32,
            )
        ),
        (
            "char",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=2,
                max_features=100_000,
                sublinear_tf=True,
                dtype=np.float32,
            )
        ),
    ])

    X_tfidf = tfidf.fit_transform(train_texts)
    X_test_tfidf = tfidf.transform(test_texts)

    max_components = min(n_components, X_tfidf.shape[1] - 1, X_tfidf.shape[0] - 1)
    if max_components < 16:
        raise ValueError(f"Слишком мало признаков для SVD: n_components={max_components}")

    svd = TruncatedSVD(
        n_components=max_components,
        n_iter=7,
        random_state=SEED,
    )

    X_svd = svd.fit_transform(X_tfidf).astype(np.float32)
    X_test_svd = svd.transform(X_test_tfidf).astype(np.float32)

    print("TF-IDF shape:", X_tfidf.shape)
    print("SVD shape:", X_svd.shape)
    print("SVD explained variance:", float(svd.explained_variance_ratio_.sum()))

    return X_svd, X_test_svd


def make_sparse_tfidf(train_texts, test_texts):
    vectorizer = FeatureUnion([
        (
            "word",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=180_000,
                sublinear_tf=True,
                dtype=np.float32,
            )
        ),
        (
            "char",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 6),
                min_df=2,
                max_features=120_000,
                sublinear_tf=True,
                dtype=np.float32,
            )
        ),
    ])

    X_train_sparse = vectorizer.fit_transform(train_texts)
    X_test_sparse = vectorizer.transform(test_texts)
    return X_train_sparse, X_test_sparse


def predict_proba_safely(model, X):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if isinstance(proba, list):
            proba = np.vstack([p[:, 1] for p in proba]).T
        return np.nan_to_num(np.asarray(proba, dtype=np.float32), nan=0.0, posinf=1.0, neginf=0.0)

    decision = model.decision_function(X)
    return sigmoid_np(decision)


def run_classic_oof(name, estimator, train_texts, test_texts):
    print("=" * 100)
    print("Classic model:", name)

    oof_probs = np.zeros((len(train), N_LABELS), dtype=np.float32)

    for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        print(f"{name} fold {fold}/{N_SPLITS}")

        X_fold, X_valid = make_sparse_tfidf(
            pd.Series(train_texts).iloc[train_idx],
            pd.Series(train_texts).iloc[valid_idx],
        )

        model = clone(estimator)
        model.fit(X_fold, Y[train_idx].astype(int))
        oof_probs[valid_idx] = predict_proba_safely(model, X_valid)

        fold_pred = (oof_probs[valid_idx] >= 0.5).astype(int)
        print_metrics(Y[valid_idx], fold_pred, title=f"{name} fold {fold} @0.5")

        del X_fold, X_valid, model
        gc.collect()

    print_metrics(Y, (oof_probs >= 0.5).astype(int), title=f"{name} OOF @0.5")

    X_full, X_test_sparse = make_sparse_tfidf(train_texts, test_texts)
    final_model = clone(estimator)
    final_model.fit(X_full, Y.astype(int))
    test_probs = predict_proba_safely(final_model, X_test_sparse)

    del X_full, X_test_sparse, final_model
    gc.collect()

    return oof_probs.astype(np.float32), test_probs.astype(np.float32)


classic_specs = [
    (
        "tfidf_logreg_meta",
        OneVsRestClassifier(
            LogisticRegression(
                C=4.0,
                solver="liblinear",
                class_weight="balanced",
                max_iter=1500,
                random_state=SEED,
            ),
            n_jobs=-1,
        ),
        X_text,
        X_test_text,
    ),
    (
        "tfidf_logreg_nometa",
        OneVsRestClassifier(
            LogisticRegression(
                C=3.0,
                solver="liblinear",
                class_weight="balanced",
                max_iter=1500,
                random_state=SEED,
            ),
            n_jobs=-1,
        ),
        X_text_nometa,
        X_test_text_nometa,
    ),
    (
        "tfidf_complement_nb",
        OneVsRestClassifier(
            ComplementNB(alpha=0.15),
            n_jobs=-1,
        ),
        X_text,
        X_test_text,
    ),
]

for name, estimator, train_texts, test_texts in classic_specs:
    oof_p, test_p = run_classic_oof(name, estimator, train_texts, test_texts)
    model_oof_probs[name] = oof_p
    model_test_probs[name] = test_p


def make_metadata_features(train_df, test_df, top_n_sources=80):
    train_meta = pd.DataFrame(index=train_df.index)
    test_meta = pd.DataFrame(index=test_df.index)

    for df, out in [(train_df, train_meta), (test_df, test_meta)]:
        title = df["title"].fillna("").astype(str)
        text = df["text"].fillna("").astype(str)

        out["title_len"] = np.log1p(title.str.len())
        out["text_len"] = np.log1p(text.str.len())
        out["title_words"] = np.log1p(title.str.split().str.len())
        out["text_words"] = np.log1p(text.str.split().str.len())
        out["title_digits"] = np.log1p(title.str.count(r"\d"))
        out["text_digits"] = np.log1p(text.str.count(r"\d"))
        out["title_exclam"] = np.log1p(title.str.count("!"))
        out["text_exclam"] = np.log1p(text.str.count("!"))
        out["title_question"] = np.log1p(title.str.count(r"\?"))
        out["text_question"] = np.log1p(text.str.count(r"\?"))

        dates = pd.to_datetime(df["publication_date"], errors="coerce")
        out["year"] = dates.dt.year.fillna(-1).astype(float)
        out["month"] = dates.dt.month.fillna(-1).astype(float)
        out["dayofweek"] = dates.dt.dayofweek.fillna(-1).astype(float)

    top_sources = (
        train_df["source"]
        .fillna("missing_source")
        .astype(str)
        .value_counts()
        .head(top_n_sources)
        .index
    )

    def map_source(series):
        source = series.fillna("missing_source").astype(str)
        return source.where(source.isin(top_sources), "__other_source__")

    train_source = pd.get_dummies(map_source(train_df["source"]), prefix="source")
    test_source = pd.get_dummies(map_source(test_df["source"]), prefix="source")
    test_source = test_source.reindex(columns=train_source.columns, fill_value=0)

    train_features = pd.concat([train_meta, train_source], axis=1).astype(np.float32)
    test_features = pd.concat([test_meta, test_source], axis=1).astype(np.float32)

    return train_features.values, test_features.values


X_svd, X_test_svd = make_tfidf_svd_features(
    X_text,
    X_test_text,
    n_components=320,
)

X_meta, X_test_meta = make_metadata_features(train, test, top_n_sources=80)


def make_corrector_matrix(model_probs_dict, svd_features, meta_features):
    blocks = []
    for name in sorted(model_probs_dict.keys()):
        probs = np.asarray(model_probs_dict[name], dtype=np.float32)
        blocks.append(probs)
        blocks.append(prob_to_logit_np(probs))

    blocks.extend([
        svd_features.astype(np.float32),
        meta_features.astype(np.float32),
    ])

    return np.hstack(blocks).astype(np.float32)


X_corrector = make_corrector_matrix(
    model_oof_probs,
    X_svd,
    X_meta,
)

X_test_corrector = make_corrector_matrix(
    model_test_probs,
    X_test_svd,
    X_test_meta,
)

print("All base components:", list(model_oof_probs.keys()))
print("Corrector train matrix:", X_corrector.shape)
print("Corrector test matrix:", X_test_corrector.shape)


## 10. MLP-корректор и финальный postprocessing

Корректор обучается по OOF-схеме на предсказаниях базовых моделей и дополнительных признаках.

Финальный ансамбль подбирается не одним общим `alpha`, а отдельно по каждому классу:

- веса моделей выбираются по OOF;
- порог выбирается точно по отсортированным OOF-score, без грубой сетки `0.05` или `0.005`;
- оптимизируется `hamming_loss`, то есть именно та величина, которую нужно уменьшать.


In [ ]:
class CorrectorDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        if self.labels is None:
            return self.features[idx]
        return self.features[idx], self.labels[idx]


class CorrectorNet(torch.nn.Module):
    def __init__(self, input_dim, hidden1=512, hidden2=256, dropout1=0.30, dropout2=0.18):
        super().__init__()

        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden1),
            torch.nn.LayerNorm(hidden1),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout1),

            torch.nn.Linear(hidden1, hidden2),
            torch.nn.LayerNorm(hidden2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout2),

            torch.nn.Linear(hidden2, N_LABELS),
        )

    def forward(self, x):
        return self.net(x)


class MLPCorrector:
    def __init__(
        self,
        input_dim,
        epochs=70,
        batch_size=256,
        lr=1.5e-3,
        weight_decay=2e-4,
        patience=8,
        pos_weight_mode="sqrt",
    ):
        self.input_dim = input_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.patience = patience
        self.pos_weight_mode = pos_weight_mode

        seed_everything(SEED)

        self.scaler = StandardScaler()
        self.model = CorrectorNet(input_dim=input_dim).to(DEVICE)
        self.loss_fn = None
        self.best_epoch_ = epochs

    def fit(self, X, y, X_valid=None, y_valid=None):
        seed_everything(SEED)

        X_scaled = self.scaler.fit_transform(X).astype(np.float32)
        X_valid_scaled = None if X_valid is None else self.scaler.transform(X_valid).astype(np.float32)

        self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=make_pos_weight(y, mode=self.pos_weight_mode))

        train_dataset = CorrectorDataset(X_scaled, y)

        generator = torch.Generator()
        generator.manual_seed(SEED)

        loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            generator=generator,
            num_workers=0,
            drop_last=False,
            pin_memory=torch.cuda.is_available(),
        )

        optimizer = make_adamw(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        best_state = None
        best_valid_loss = np.inf
        bad_epochs = 0

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            train_losses = []

            for xb, yb in loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                logits = self.model(xb)
                loss = self.loss_fn(logits, yb)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()

                train_losses.append(loss.item())

            if X_valid_scaled is None:
                continue

            valid_loss = self.evaluate_loss(X_valid_scaled, y_valid)

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in self.model.state_dict().items()
                }
                self.best_epoch_ = epoch
                bad_epochs = 0
            else:
                bad_epochs += 1

            if bad_epochs >= self.patience:
                break

        if best_state is not None:
            self.model.load_state_dict(best_state)
            del best_state

        return self

    def evaluate_loss(self, X_scaled, y):
        dataset = CorrectorDataset(X_scaled, y)
        loader = DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )

        self.model.eval()
        losses = []

        with torch.no_grad():
            for xb, yb in loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)

                logits = self.model(xb)
                loss = self.loss_fn(logits, yb)
                losses.append(loss.item())

        return float(np.mean(losses))

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X).astype(np.float32)
        dataset = CorrectorDataset(X_scaled, labels=None)
        loader = DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )

        self.model.eval()
        all_probs = []

        with torch.no_grad():
            for xb in loader:
                xb = xb.to(DEVICE, non_blocking=True)
                logits = self.model(xb)
                probs = torch.sigmoid(logits).detach().cpu().numpy()
                all_probs.append(probs.astype(np.float32))

        return np.vstack(all_probs)


def best_threshold_for_binary(y_true_binary, scores):
    y_true_binary = np.asarray(y_true_binary, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float32)

    order = np.argsort(-scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_y = y_true_binary[order]

    total_pos = int(sorted_y.sum())
    fp = 0
    fn = total_pos

    best_errors = fp + fn
    best_thr = np.float32(1.001)  # выше любого score: все предсказания 0

    n = len(sorted_y)
    i = 0
    while i < n:
        score_value = sorted_scores[i]
        j = i + 1
        while j < n and sorted_scores[j] == score_value:
            j += 1

        group_pos = int(sorted_y[i:j].sum())
        group_size = j - i
        group_neg = group_size - group_pos

        fn -= group_pos
        fp += group_neg
        errors = fp + fn

        if errors < best_errors:
            best_errors = errors
            best_thr = np.float32(score_value)

        i = j

    return best_thr, best_errors / len(sorted_y)


def nested_threshold_loss(y_label, scores, splits):
    """
    Для каждого fold порог подбирается на остальных OOF-объектах и применяется к holdout-fold.
    Это менее агрессивно, чем подбирать один идеальный threshold на всём OOF.
    """
    y_label = np.asarray(y_label, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float32)

    pred = np.zeros_like(y_label, dtype=np.int8)
    fold_thresholds = []

    all_idx = np.arange(len(y_label))

    for _, valid_idx in splits:
        valid_mask = np.zeros(len(y_label), dtype=bool)
        valid_mask[valid_idx] = True
        train_for_thr_idx = all_idx[~valid_mask]

        thr, _ = best_threshold_for_binary(y_label[train_for_thr_idx], scores[train_for_thr_idx])
        fold_thresholds.append(float(thr))
        pred[valid_idx] = (scores[valid_idx] >= thr).astype(np.int8)

    loss = np.mean(pred != y_label)
    final_thr = float(np.median(fold_thresholds))
    return final_thr, float(loss), fold_thresholds


def generate_weight_candidates(component_names, n_random=600):
    rng = np.random.default_rng(SEED)
    n_components = len(component_names)
    candidates = []

    # Отдельные модели.
    for i in range(n_components):
        w = np.zeros(n_components, dtype=np.float32)
        w[i] = 1.0
        candidates.append(w)

    # Простые устойчивые средние.
    candidates.append(np.ones(n_components, dtype=np.float32) / n_components)

    groups = [
        [i for i, name in enumerate(component_names) if name.startswith("tr_")],
        [i for i, name in enumerate(component_names) if name.startswith("tfidf_")],
        [i for i, name in enumerate(component_names) if (name.startswith("tr_") or name == "mlp_corrector")],
        [i for i, name in enumerate(component_names) if (name.startswith("tr_") or name.startswith("tfidf_logreg"))],
    ]

    for group in groups:
        if group:
            w = np.zeros(n_components, dtype=np.float32)
            w[group] = 1.0 / len(group)
            candidates.append(w)

    # Регуляризованные случайные веса: меньше экстремальных вариантов, чтобы не переобучаться на OOF.
    alphas = [0.75, 1.5, 3.0, 6.0]
    per_alpha = max(1, n_random // len(alphas))

    for alpha in alphas:
        sampled = rng.dirichlet(np.ones(n_components) * alpha, size=per_alpha).astype(np.float32)

        # Мягкое ограничение: ансамбль не должен целиком проваливаться в слабые классические модели.
        for w in sampled:
            tr_weight = sum(w[i] for i, name in enumerate(component_names) if name.startswith("tr_"))
            if tr_weight >= 0.35:
                candidates.append(w)

    candidates = np.vstack(candidates).astype(np.float32)
    candidates = candidates / candidates.sum(axis=1, keepdims=True)
    candidates = np.unique(np.round(candidates, 5), axis=0)
    return candidates


def make_labelwise_blend_scores(probs_dict, component_names, weights_by_label):
    n_rows = next(iter(probs_dict.values())).shape[0]
    scores = np.zeros((n_rows, N_LABELS), dtype=np.float32)

    for label_idx in range(N_LABELS):
        for model_idx, name in enumerate(component_names):
            scores[:, label_idx] += weights_by_label[label_idx, model_idx] * probs_dict[name][:, label_idx]

    return scores.astype(np.float32)


def search_labelwise_ensemble(oof_probs_dict, y_true, n_random=600):
    component_names = sorted(oof_probs_dict.keys())
    candidates = generate_weight_candidates(component_names, n_random=n_random)

    weights_by_label = np.zeros((N_LABELS, len(component_names)), dtype=np.float32)
    thresholds = np.zeros(N_LABELS, dtype=np.float32)
    rows = []

    for label_idx in range(N_LABELS):
        label_matrix = np.vstack([
            np.asarray(oof_probs_dict[name][:, label_idx], dtype=np.float32)
            for name in component_names
        ])
        y_label = y_true[:, label_idx].astype(int)

        best_loss = np.inf
        best_thr = 0.5
        best_w = None
        best_fold_thresholds = None

        for w in candidates:
            scores = np.dot(w, label_matrix)
            thr, loss, fold_thresholds = nested_threshold_loss(y_label, scores, cv_splits)

            if loss < best_loss:
                best_loss = loss
                best_thr = thr
                best_w = w.copy()
                best_fold_thresholds = fold_thresholds

        # После выбора устойчивых весов финальный threshold слегка уточняется на всём OOF.
        # Если общий threshold заметно расходится с fold-wise median, оставляем median как более устойчивый.
        final_scores = np.dot(best_w, label_matrix)
        full_thr, full_loss = best_threshold_for_binary(y_label, final_scores)
        if abs(float(full_thr) - float(best_thr)) <= 0.04:
            selected_thr = float(full_thr)
            selected_loss = float(full_loss)
            threshold_mode = "full_oof"
        else:
            selected_thr = float(best_thr)
            selected_loss = float(np.mean((final_scores >= selected_thr).astype(int) != y_label))
            threshold_mode = "fold_median"

        weights_by_label[label_idx] = best_w
        thresholds[label_idx] = selected_thr

        row = {
            "label": f"label_{label_idx}",
            "threshold": float(selected_thr),
            "label_hamming_loss": float(selected_loss),
            "nested_label_hamming_loss": float(best_loss),
            "threshold_mode": threshold_mode,
            "fold_thresholds": np.round(best_fold_thresholds, 4).tolist(),
        }
        for model_idx, name in enumerate(component_names):
            row[f"w_{name}"] = float(best_w[model_idx])
        rows.append(row)

    summary = pd.DataFrame(rows)
    return component_names, weights_by_label, thresholds, summary



corrector_oof_probs = np.zeros((len(train), N_LABELS), dtype=np.float32)
corrector_best_epochs = []

for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print("=" * 80)
    print(f"MLP-corrector fold {fold}/{N_SPLITS}")

    corrector = MLPCorrector(
        input_dim=X_corrector.shape[1],
        epochs=70,
        batch_size=256,
        lr=1.5e-3,
        weight_decay=2e-4,
        patience=8,
        pos_weight_mode="sqrt",
    )

    corrector.fit(
        X_corrector[train_idx],
        Y[train_idx],
        X_valid=X_corrector[valid_idx],
        y_valid=Y[valid_idx],
    )

    corrector_oof_probs[valid_idx] = corrector.predict_proba(X_corrector[valid_idx])
    corrector_best_epochs.append(corrector.best_epoch_)

    fold_pred = (corrector_oof_probs[valid_idx] >= 0.5).astype(int)
    print("best_epoch:", corrector.best_epoch_)
    print_metrics(Y[valid_idx], fold_pred, title=f"MLP-corrector fold {fold} @0.5")

    del corrector
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert not np.isnan(corrector_oof_probs).any()
print_metrics(Y, (corrector_oof_probs >= 0.5).astype(int), title="MLP-corrector OOF @0.5")

ensemble_oof_probs = dict(model_oof_probs)
ensemble_oof_probs["mlp_corrector"] = corrector_oof_probs

component_names, weights_by_label, thresholds, weight_summary = search_labelwise_ensemble(
    ensemble_oof_probs,
    Y,
    n_random=600,
)

display(weight_summary)

best_oof_scores = make_labelwise_blend_scores(
    ensemble_oof_probs,
    component_names,
    weights_by_label,
)

oof_pred_tuned = (best_oof_scores >= thresholds.reshape(1, -1)).astype(int)

print("Final ensemble components:", component_names)
print("Thresholds:", thresholds)
print_metrics(Y, oof_pred_tuned, title="Best label-wise ensemble OOF result")

final_corrector_epochs = int(np.median(corrector_best_epochs))
final_corrector_epochs = max(15, final_corrector_epochs)
print("Final corrector epochs:", final_corrector_epochs)

final_corrector = MLPCorrector(
    input_dim=X_corrector.shape[1],
    epochs=final_corrector_epochs,
    batch_size=256,
    lr=1.5e-3,
    weight_decay=2e-4,
    patience=8,
    pos_weight_mode="sqrt",
)

final_corrector.fit(X_corrector, Y)
corrector_test_probs = final_corrector.predict_proba(X_test_corrector)

ensemble_test_probs = dict(model_test_probs)
ensemble_test_probs["mlp_corrector"] = corrector_test_probs

test_scores = make_labelwise_blend_scores(
    ensemble_test_probs,
    component_names,
    weights_by_label,
)

test_pred = (test_scores >= thresholds.reshape(1, -1)).astype(int)

diagnostics_df = pd.DataFrame({
    "label": label_cols,
    "threshold": thresholds,
    "train_positive_rate": Y.mean(axis=0),
    "oof_positive_rate": oof_pred_tuned.mean(axis=0),
    "test_positive_rate": test_pred.mean(axis=0),
})

display(diagnostics_df)

print("test_scores shape:", test_scores.shape)
print("test_pred shape:", test_pred.shape)


## 11. Формирование submission

`id` берётся из `sample_submission`, чтобы сохранить требуемый порядок строк.  
Итоговый файл называется строго `sample_submission.csv`.



In [ ]:
def make_submission_from_pred(pred, filename="sample_submission.csv"):
    pred = np.asarray(pred).astype(int)

    assert pred.shape == (len(test), 5), f"Неверная форма pred: {pred.shape}"
    assert set(np.unique(pred)).issubset({0, 1}), "В pred должны быть только 0/1"

    pred_df = pd.DataFrame(pred, columns=label_cols)
    pred_df["id"] = test["id"].values

    sub = sample_submission[["id"]].copy()
    sub = sub.merge(pred_df, on="id", how="left")

    assert sub[label_cols].isna().sum().sum() == 0, "После merge появились пропуски"

    def format_target(row):
        values = [int(row[f"label_{i}"]) for i in range(5)]
        return "[" + ",".join(map(str, values)) + "]"

    sub["target"] = sub.apply(format_target, axis=1)
    sub = sub[["id", "target"]]

    sub.to_csv(filename, index=False)

    print(f"Файл {filename} сохранён.")
    print("Путь:", os.path.abspath(filename))
    display(sub.head())
    print(sub.shape)

    return sub


submission = make_submission_from_pred(test_pred, "sample_submission.csv")

try:
    from google.colab import files
    files.download("sample_submission.csv")
except ImportError:
    print("Автоскачивание доступно только в Google Colab.")



## 12. Финальная проверка файла



In [ ]:
check = pd.read_csv("sample_submission.csv")

assert list(check.columns) == ["id", "target"]
assert len(check) == len(sample_submission)
assert check["id"].tolist() == sample_submission["id"].tolist()

parsed_check = check["target"].apply(parse_target)
check_matrix = np.array(parsed_check.tolist(), dtype=int)

assert check_matrix.shape == (len(sample_submission), 5)
assert set(np.unique(check_matrix)).issubset({0, 1})

assert check["target"].str.match(r"^\[[01],[01],[01],[01],[01]\]$").all()

print("sample_submission.csv корректен.")
display(check.head())

## 13. Итоговая схема

В финальном решении используется честная воспроизводимая схема без внешних данных:

1. Предобработка текста: очистка `title` и `text`; для transformer используется формат `title + начало текста + конец текста`, чтобы не терять информацию из длинных материалов.
2. Cross-validation: 5 folds с фиксированным seed `322`.
3. Нейросеточная часть:
   - `ai-forever/ruRoberta-large`;
   - `FacebookAI/xlm-roberta-large`.
4. Для обеих нейросетей используется Asymmetric Loss, более подходящий для multilabel-задач с дисбалансом меток.
5. Классические ГО/ML-модели:
   - TF-IDF word/char + Logistic Regression;
   - TF-IDF без meta-признаков;
   - TF-IDF + ComplementNB.
6. MLP-корректор обучается на OOF-предсказаниях базовых моделей, SVD-компонентах TF-IDF и metadata-признаках.
7. Финальный ансамбль строится label-wise: для каждого класса отдельно выбираются веса моделей.
8. Thresholds подбираются устойчиво: для каждого fold threshold подбирается на остальных OOF-объектах и применяется к holdout-fold, затем используется стабильный fold-wise median / full OOF threshold при малом расхождении.
9. На выходе формируется `sample_submission.csv`.

Такой postprocessing не использует ответы test и не подбирается под public leaderboard.
